# Verify the final NiyamTrace-X manuscript results
Upload the complete redrafted-paper ZIP, then run the cell below. This recomputes retained-record results and checks synthetic SQLite effects. It does not run a model or fix the native polarity failure. Use a CPU runtime; no model ID, token or repository import is needed.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, tempfile, subprocess, sys

uploaded = files.upload()
archives = [Path(n) for n in uploaded if n.endswith('.zip')]
if len(archives) != 1:
    raise ValueError("Upload one NTX_Papers_Redrafted_2026-10-13.zip file.")
destination = Path(tempfile.mkdtemp(prefix="ntx_paper_verify_"))
with zipfile.ZipFile(archives[0]) as z:
    for member in z.infolist():
        p = Path(member.filename)
        if p.is_absolute() or ".." in p.parts or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Unsafe archive member: " + member.filename)
    z.extractall(destination)
scripts = list(destination.rglob("Analysis/reproduce_results.py"))
if len(scripts) != 1:
    raise ValueError("The archive does not contain exactly one analysis entry point.")
for script_name in ["reproduce_results.py", "reproduce_history.py"]:
    result = subprocess.run([sys.executable, str(scripts[0].parent / script_name)], text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError("Verification failed in " + script_name + ". Keep the full output for diagnosis.")
print("Finished: retained paper results verified. No GPU or model ID was needed.")
